## MO CapacityState

In [ ]:
import pandas as pd

CapacityState= pd.read_csv('C:/Dumps/LTE_Total/CapacityState.txt', sep='\t', usecols=['UF', 'eNB', 'capacityStateId', 'grantedCapacityLevel',\
                           'licensedCapacityLimitReached', 'licenseState', 'serviceState', 'description'])

CapacityState1= CapacityState[CapacityState['capacityStateId']=='CXC4012367'] 

CapacityState1.shape

# Calculando a banda total existente em cada eNB

In [ ]:
import pandas as pd
from datetime import datetime

semana=datetime.today().strftime('%U')
semana_atual= int(semana) + 1
print(semana_atual)

arquivo_csv= 'C:/Camilo/' + 'EUtranCell_TDD_FDD'+ '_'+str(semana_atual)+'.txt'

df= pd.read_csv(arquivo_csv,sep='\t',engine='python', encoding='ISO-8859-1')

# Supondo que 'df' é seu DataFrame
df['Setor'] = df['CELL'].str[-1] # Extraia o último caractere da coluna CELL

df['dlChannelBandwidth'] = df['dlChannelBandwidth'].astype(float)/1000 # Certifique-se de que dlChannelBandWidth é float

# Agora, agrupamos por UF, MUNICIPIO, eNB e Setor, depois somamos dlChannelBandWidth e contamos as células
df2 = df.groupby(['UF', 'MUNICIPIO', 'eNB', 'Setor']).agg(Banda=('dlChannelBandwidth', 'sum'), conta_celulas=('CELL', 'count')).reset_index()

df2['Banda_Total_Utilizada_MHz'] = df2.groupby('eNB')['Banda'].transform('sum')

# Exibir o dataframe resultante
print(df2.head())

#df2.to_excel('C:/Camilo/teste3.xlsx', index=None)


In [ ]:
df3= df2.groupby(["UF", "MUNICIPIO", "eNB"])["Banda_Total_Utilizada_MHz"].max().reset_index()

df3.head()

## Juntando o MO CapacityState com os dados de Banda por eNB

In [ ]:
junta= pd.merge(CapacityState1, df3, left_on=['UF', 'eNB'], right_on=['UF', 'eNB'], how='left')

junta['Sobra_de_Licença_MHz']= junta['grantedCapacityLevel'] * 5 - junta['Banda_Total_Utilizada_MHz']

junta['Regional']= junta['UF'].replace(['AM', 'AP', 'MA', 'PA', 'RR', 'BA', 'SE'], ['N','N','N','N','N','BASE', 'BASE'])

junta= junta[['Regional','MUNICIPIO','UF',
 'eNB',
 'capacityStateId',
 'description',
 'grantedCapacityLevel',
 'licensedCapacityLimitReached',
 'licenseState',
 'serviceState','Banda_Total_Utilizada_MHz',
 'Sobra_de_Licença_MHz']]

junta1= junta[junta['eNB'].str[:1].isin(['T', 'M'])]

junta.head()

junta1.to_excel('C:/Vivo Reports/LTE/LTE Parametros/Audit_CapacityState.xlsx', index=None)
print('Done')